### ~

In [34]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from pathlib import Path
import shutil
import subprocess
import builtins

ROOT = Path.cwd()

def here():
    return ROOT

def make_dir(path):
    p = ROOT / path
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_file(path, content=""):
    p = ROOT / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return p

def remove_path(path):
    p = ROOT / path
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()
    return p

def run_cmd(*cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

if __name__ == "__main__":
    print("This is a utility module. Please import and use the functions as needed.")
    make_dir("Helpers")
    make_file("Helpers/__init__.py",'''
import builtins
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Store the original input function once, globally, to avoid re-capturing a mocked input
if not hasattr(builtins, '_original_input_backup'):
    builtins._original_input_backup = builtins.input

def set_test_inputs(inputs_list):
    """
    Sets up a mock input function that draws from a list of inputs.
    If inputs_list is None or empty, restores the original input function.
    """
    if inputs_list:
        input_iter = iter(inputs_list)
        def mock_input(prompt=""):
            try:
                value = next(input_iter)
                print(f"{prompt}{value}")
                return value
            except StopIteration:
                raise EOFError("No more inputs for testing")
        builtins.input = mock_input
    else:
        builtins.input = builtins._original_input_backup

# Define test_inputs, this is where you can change your desired inputs
test_inputs = [1]  # <--- Change this list to provide your test inputs

# Apply the test inputs automatically when this cell is run
set_test_inputs(test_inputs)

def setin(*inputs):
    """
    A helper function to set test inputs for the input() function.
    Usage:
    setin("input1", "input2", "input3")
    This will set up the input() function to return "input1" on the first call,
    "input2" on the second call, and so on.
    To reset to normal input behavior, call setin() with no arguments or None:
    setin()
    """
    if inputs:
        input_iter = iter(inputs)
        def mock_input(prompt=""):
            try:
                value = next(input_iter)
                print(f"{prompt}{value}")
                return value
            except StopIteration:
                raise EOFError("No more inputs for testing")
        builtins.input = mock_input
    else:
        builtins.input = builtins._original_input_backup
              ''')
import importlib
import sys

# Remove cached module if it exists
if 'Helpers' in sys.modules:
    del sys.modules['Helpers']

# Now import fresh
import Helpers
from Helpers import setin

setin("Hello", "World")

import os
from IPython.display import display, Javascript

from pathlib import Path
import shutil
import subprocess


class WorkspaceBase:
    @staticmethod
    def get_notebook_name():
        notebook_files = [f for f in os.listdir('.') if f.endswith('.ipynb')]
        if notebook_files:
            thienotebooktitleworkspace = os.path.splitext(notebook_files[0])[0]
        else:
            thienotebooktitleworkspace = "Untitled Notebook" # Default if no .ipynb files are found
        return thienotebooktitleworkspace
    


    def get_root(self):
        return self.root

    def make_folder(self, name):
        folder = self.root / name
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def write_text(self, path, text):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        return path

    def read_text(self, path):
        return Path(path).read_text(encoding="utf-8")


    def run_cmd(self, *args):
        result = subprocess.run(args, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)
    def __init__(self, root=None):
        if root is None:
            root = f"{self.get_notebook_name()}_workspace"
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

class Problem_(WorkspaceBase):
    def __init__(self, name, files=None, root=None):
        super().__init__(root=root)
        self.name = name
        self.files = files or {}
        self.folder = self.make_folder(self.safe_name())

    def safe_name(self):
        return self.name.strip().lower().replace(" ", "_")

    def add_file(self, relative_path, content):
        self.files[str(relative_path)] = content

    def get_file(self, relative_path):
        return self.files.get(str(relative_path))

    def remove_file(self, relative_path):
        self.files.pop(str(relative_path), None)

    def save(self, clear_first=False):
        if clear_first:
            self.reset_folder(self.folder)

        for relative_path, content in self.files.items():
            full_path = self.folder / relative_path
            self.write_text(full_path, content)
    def run(self, command, cwd=None):
        if cwd is None:
            cwd = self.folder
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True, shell=True)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)
    def reset_folder(self, folder):
        folder = Path(folder)
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True, exist_ok=True)
        return folder
    
    def load(self):
        loaded = {}
        for relative_path in self.list_all_files(self.folder):
            full_path = self.folder / relative_path
            loaded[relative_path] = self.read_text(full_path)
        self.files = loaded
        return loaded

    def summary(self):
        return {
            "name": self.name,
            "folder": str(self.folder),
            "file_count": len(self.files),
            "files": sorted(self.files.keys())
        }

    def __repr__(self):
        return f"Problem(name={self.name!r}, files={len(self.files)}, folder={str(self.folder)!r})"
    
class Problem(Problem_):
    def __init__(self, name, files=None, root=None):
        super().__init__(name, files, root)
        #self.run_cmd("ls", "-la", str(self.folder))
        self.reset_folder(self.folder)

    def runpy(self, command, cwd=None):
        print(f"Ran command: python {command} in {cwd or self.folder}")
        self.run(f"python {command}", cwd)

    
    def create(self, relative_path, content):
        self.add_file(relative_path, content)
        self.save()
        print(f"Created file: {relative_path} with content:\n{content}")
    def create2(self, relative_path, content):
        self.create(relative_path, content)
        self.runpy(relative_path, cwd=self.folder)
    def answer(self, content):
        self.add_file("ANSWER.md", content)
        self.save()
        print(f"Saved answer to ANSWER.md:\n{content}")
        


This is a utility module. Please import and use the functions as needed.


PosixPath('/workspaces/cs2520-study/zybooks notebooks/C10/Helpers')

PosixPath('/workspaces/cs2520-study/zybooks notebooks/C10/Helpers/__init__.py')

### ~

In [35]:
'''11.1 Modules
Module basics
The interactive Python interpreter provides the most basic way to execute Python code. However, all of the defined variables, functions, classes, etc., are lost when a programmer closes the interpreter. Thus, a programmer will typically write Python code in a file, and then pass that file as input to the interpreter. Such a file is called a script.

A programmer may find themselves writing the same function over and over again in multiple scripts, or creating very long and difficult-to-maintain scripts. A solution is to use a module, which is a file containing Python code that can be imported and used by scripts, other modules, or the interactive interpreter. To import a module means to execute the code contained by the module and make the definitions within that module available for use by the importing program.

participation activity
11.1.1: A module is a file containing Python statements and definitions that can be used by other Python sources.


1

2

The functions can instead be defined in another file. The file can be imported as a "module".
def fct():
   # ...
def sq():   
   # ...
x = fct() * sq()
# ...
script1.py
def fct():
   # ...
def sq():   
   # ...
x = fct() / sq()
# ...
script2.pyfuncs.py
def fct():
   # ...
def sq():   
   # ...
script1.py
import funcs
script2.py
import funcs
x = funcs.fct() * 
      funcs.sq()
x = funcs.fct() / 
      funcs.sq()
fct()sq()
*
fct()sq()
/
def fct():
   # ...
def sq():   
   # ...
Static figure: Begin Python code (script1.py, before using a module): def fct(): # ... def sq(): # ... x = fct() * sq() # ... End Python code. Begin Python code (script2.py, before using a module): def fct(): # ... def sq(): # ... x = fct() / sq() # ... End Python code. Begin Python code (funcs.py module): def fct(): # ... def sq(): # ... End Python code. Begin Python code (script1.py, after using a module): import funcs x = funcs.fct() * funcs.sq() End Python code. Begin Python code (script2.py, after using a module): import funcs x = funcs.fct() / funcs.sq() End Python code. Step 1: A programmer writes scripts containing functions and code using those functions. Multiple scripts might define the same functions. script1.py and script2.py are shown defining the same functions fct() and sq(). Step 2: The functions can instead be defined in another file. The file can be imported as a "module". The module funcs.py is created with the same function definitions in both script1.py and script2.py. script1.py is rewritten to instead import the functions from the funcs.py module. script1.py is executed, starting with the import statement. To execute the calculation for x, the function names from the module are shown above the function names in script1.py. The process repeats for script2.py using the same module functions, but with a different calculation for x.

Captions
A programmer writes scripts containing functions and code using those functions. Multiple scripts might define the same functions.
The functions can instead be defined in another file. The file can be imported as a "module".
Playing step 2: The functions can instead be defined in another file. The file can be imported as a "module". Step finished playing

Feedback?

A module's filename should end with ".py"; otherwise, the interpreter will not be able to import the module. The module_name item should match the filename of the module, but without the .py extension. Ex: If a programmer wants to import a module whose filename is HTTPServer.py, the import statement import HTTPServer would be used. Note that import statements are case-sensitive; thus, import ABC is distinct from import aBc.

The interpreter must also be able to find the module to import. The simplest solution is to keep modules in the same directory as the executing script; however, the interpreter can also search the computer's file system for the modules. Later material covers these search mechanisms.

Good practice is to place import statements at the top of a file. There are few useful instances of placing import statements in any location other than the top. The benefit of placing import statements at the top is that a reader of the program can quickly identify the modules required for the program to run. A module being required by another program is often called a dependency.'''

_= Problem("participation activity 11.1.1")
_.create("script1.py", '''import funcs
x = funcs.fct() * funcs.sq()''')
_.create("funcs.py", '''def fct():
    return 42
def sq():
    return 7''')
_.runpy("script1.py", cwd=_.folder)


'11.1 Modules\nModule basics\nThe interactive Python interpreter provides the most basic way to execute Python code. However, all of the defined variables, functions, classes, etc., are lost when a programmer closes the interpreter. Thus, a programmer will typically write Python code in a file, and then pass that file as input to the interpreter. Such a file is called a script.\n\nA programmer may find themselves writing the same function over and over again in multiple scripts, or creating very long and difficult-to-maintain scripts. A solution is to use a module, which is a file containing Python code that can be imported and used by scripts, other modules, or the interactive interpreter. To import a module means to execute the code contained by the module and make the definitions within that module available for use by the importing program.\n\nparticipation activity\n11.1.1: A module is a file containing Python statements and definitions that can be used by other Python sources.\n\

Created file: script1.py with content:
import funcs
x = funcs.fct() * funcs.sq()
Created file: funcs.py with content:
def fct():
    return 42
def sq():
    return 7
Ran command: python script1.py in driver_workspace/participation_activity_11.1.1



In [37]:
'''participation activity
11.1.2: Basic importing of modules.
1)
A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
>>>

Check

Show answer
2)
A file containing Python code that is passed as input to the interpreter is called a _______.

Check

Show answer
3)
A _______ is a file containing Python code that can be imported by a script.

Check

Show answer

Feedback?'''

_ = Problem("participation activity 11.1.2")
_.create("tools.py", '''def greet(name):
    return f"Hello, {name}!"''')
_.runpy("tools.py", cwd=_.folder)
_.create("script.py", '''import tools
print(tools.greet("Alice"))''')
_.runpy("script.py", cwd=_.folder)
_.create("ANSWER.md", '''# Participation Activity 11.1.2
          1) A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
            >>> import tools
            2) A file containing Python code that is passed as input to the interpreter is called a _______.
            Answer: script
            3) A _______ is a file containing Python code that can be imported by a script.
            Answer: module''')

'participation activity\n11.1.2: Basic importing of modules.\n1)\nA programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.\n>>>\n\nCheck\n\nShow answer\n2)\nA file containing Python code that is passed as input to the interpreter is called a _______.\n\nCheck\n\nShow answer\n3)\nA _______ is a file containing Python code that can be imported by a script.\n\nCheck\n\nShow answer\n\nFeedback?'

Created file: tools.py with content:
def greet(name):
    return f"Hello, {name}!"
Ran command: python tools.py in driver_workspace/participation_activity_11.1.2

Created file: script.py with content:
import tools
print(tools.greet("Alice"))
Ran command: python script.py in driver_workspace/participation_activity_11.1.2
Hello, Alice!

Created file: ANSWER.md with content:
# Participation Activity 11.1.2
          1) A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
            >>> import tools
            2) A file containing Python code that is passed as input to the interpreter is called a _______.
            Answer: script
            3) A _______ is a file containing Python code that can be imported by a script.
            Answer: module


In [38]:
'''Importing a module
Evaluating an import statement initiates the following process to load the module:

A check is conducted to determine whether the module has already been imported. If already imported, then the loaded module is used.
If not already imported, a new module object is created and inserted in sys.modules.
The code in the module is executed in the new module object's namespace.
When importing a module, the interpreter first checks to see if that module has already been imported. A dictionary of the loaded modules is stored in sys.modules (available from the sys standard library module). If the module has not yet been loaded, then a new module object is created. A module object is simply a namespace that contains definitions from the module. If the module has already been loaded, then the existing module object is used.

If a module is not found in sys.modules, then the module is added and the statements within the module's code are executed. Definitions in the module's code, such as variable assignments and function definitions, are placed in the module's namespace. The module is then added to the importing script or module's namespace, so the importer can access the definitions. The below animation illustrates.

participation activity
11.1.3: Importing a module.


1

2

3

4

5

webpage has already been imported. Existing module is loaded.
import HTTPServer
import webpage

my_ip = HTTPServer.address

webpage.disp(my_ip)

# ...
sys.modules
HTTPServer.py
webpage.py
empty
HTTPServer
HTTPServer
import webpage

address = " "

# ...
def disp(ip):
   # ...

# ...
webpage
webpage
disp
webpage
address
webpage
Static figure: Begin Python code: import HTTPServer import webpage my_ip = HTTPServer.address webpage.disp(my_ip) # ... End Python code. Begin Python code (HTTPServer.py): import webpage address = " " # ... End Python code. Begin Python code (webpage.py): def disp(ip): # ... # ... End Python code. The sys.modules namespace contains HTTPServer and webpage. The HTTPServer and webpage namespaces are shown with a line drawn to their respective modules in sys.modules. The HTTPServer namespace contains webpage and address. The webpage namespace contains disp. Step 1: sys.modules checks for HTTPServer. A new module object is created. The module is then inserted into sys.modules. sys.modules is initially empty. The main Python code executes the line "import HTTPServer". sys.modules is checked for HTTPServer, a new module is created, and HTTPServer is added to sys.modules. The HTTPServer namespace is initially empty with a line drawn to HTTPServer in sys.modules. Step 2: HTTPServer's code is executed in module namespace. sys.modules checks for webpage. The new module object is created and inserted in sys.modules. The HTTPServer code executes the line "import webpage". sys.modules is checked for containing webpage, a new module is created, and webpage is added to sys.modules. The webpage namespace is initially empty with a line drawn to webpage in sys.modules. Step 3: webpage's code is executed in module namespace. webpage is added to HTTPServer namespace. webpage's code executes, and disp is defined and added to webpage's namespace. When webpage's code finishes executing, webpage is added to HTTPServer's namespace. Step 4: HTTPServer's code continues executing. address is defined and added to HTTPServer's namespace. Step 5: webpage has already been imported. Existing module is loaded. The main Python code continues and executes the next line "import webpage". sys.modules is checked for webpage, and the existing module is loaded.

Captions
sys.modules checks for HTTPServer. A new module object is created. The module is then inserted into sys.modules.
HTTPServer's code is executed in module namespace. sys.modules checks for webpage. The new module object is created and inserted in sys.modules.
webpage's code is executed in module namespace. webpage is added to HTTPServer namespace.
HTTPServer's code continues executing.
webpage has already been imported. Existing module is loaded.
Playing step 5: webpage has already been imported. Existing module is loaded. Step finished playing

Feedback?
Executing import HTTPServer causes a new module object to be created and added to sys.modules. The code of HTTPServer is executed, which contains another import statement import webpage. Since webpage has not yet been imported, a second new module object is created and added to sys.modules. Execution of the webpage code occurs, which defines a function within the webpage module's namespace. Once the webpage module is successfully imported, the execution of HTTPServer's code continues, creating new definitions in the HTTPServer module's namespace. If the script attempts to import webpage, the already created module object is used.

participation activity
11.1.4: The importing process.
Order the events as they occur when the statement import HTTPServer executes, assuming HTTPServer has not been previously imported.

How to use this tool
HTTPServer added to importer's namespace.
HTTPServer added to sys.modules.
Module object created.
sys.modules checked for HTTPServer.
HTTPServer code executed.
1st event
2nd event
3rd event
4th event
5th event

Reset

Feedback?
'''
_ = Problem("participation activity 11.1.4")
_.create("HTTPServer.py", '''import webpage
address = " "# ...''')
_.create("webpage.py", '''def disp(ip):
    print(f"Displaying webpage for IP: {ip}")
# ...''')
_.runpy("HTTPServer.py", cwd=_.folder)


'Importing a module\nEvaluating an import statement initiates the following process to load the module:\n\nA check is conducted to determine whether the module has already been imported. If already imported, then the loaded module is used.\nIf not already imported, a new module object is created and inserted in sys.modules.\nThe code in the module is executed in the new module object\'s namespace.\nWhen importing a module, the interpreter first checks to see if that module has already been imported. A dictionary of the loaded modules is stored in sys.modules (available from the sys standard library module). If the module has not yet been loaded, then a new module object is created. A module object is simply a namespace that contains definitions from the module. If the module has already been loaded, then the existing module object is used.\n\nIf a module is not found in sys.modules, then the module is added and the statements within the module\'s code are executed. Definitions in the m

Created file: HTTPServer.py with content:
import webpage
address = " "# ...
Created file: webpage.py with content:
def disp(ip):
    print(f"Displaying webpage for IP: {ip}")
# ...
Ran command: python HTTPServer.py in driver_workspace/participation_activity_11.1.4

